<div align="right"><sub>Notebook 最終更新: 2026-03-25 19:37</sub></div>
<h1><strong>05. AIエージェントによる自己改善（Iterative Refinement）</strong></h1>

今回からは、LLMを単体で使うのではなく、役割を持った「エージェント」として組み合わせて、複雑なタスクをこなす方法を学びます。

**Qwen3-8B のような現代の強力なモデルは「指示漏れ」や「フォーマット違反」のような単純なミスをほとんどしません。** 
そこでこの回では、エージェントの目的を「間違い探し」から**「さらなる品質向上（壁打ちによる自己改善）」**へとステップアップさせます。

回答を行う **Writer（執筆者）** と、それをレビューして改善要求を出す **Editor（編集長）** の2役を組み合わせたループを体験しましょう。

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. チャット関数の準備**

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 768, temp: float = 0.5):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

## **2. 執筆者と編集長のエージェント構築**
Writer（Executor）が書いた初稿に対して、Editor（Critic）が必ず「具体例の追加」と「トーンの変更」を要求し、Writerがそれに答えて書き直すプロセスを実行します。

In [ ]:
writer_prompt = """
あなたはプロのライターです。ユーザーからのテーマについて、まずは標準的な解説記事を書いてください。
もし、Editor（編集長）から修正指示が来た場合は、そのフィードバックを全面的に取り入れて、より魅力的な記事に書き直してください。
"""

editor_prompt = """
あなたは非常に厳しい編集長です。
これまでの対話履歴を確認し、あなたが『まだ一度もフィードバックを出していない（初回）』場合は、どんなに良い文章であっても必ず以下の2点を指摘して書き直しを命じてください：
1. 「読者が日常生活でイメージしやすいように、具体的な活用シーンを1つ追加してください」
2. 「全体的にもっとワクワクするような、感情豊かなトーンに書き換えてください」

もし、すでにあなたがフィードバックを出しており、ライターがそれを反映した「第2稿」を提出している場合は、その努力を認めて「誤りなし」とだけ出力し、承認してください。
"""

writer = RoleConfig(name="Writer", system_prompt=writer_prompt)
editor = RoleConfig(name="Editor", system_prompt=editor_prompt)

agent = LLMExecutorCriticAgent(llm_chat, role_configs=[writer, editor])

query = "未来の交通手段である「空飛ぶクルマ（eVTOL）」が普及した社会について、300字程度で解説記事を書いてください。"

# 最大2イテレーション（初稿発行 → 編集長指摘 → 第2稿発行 → 編集長承認）で実行します
final_answer, full_log, steps = agent.run_pipeline(query, max_iterations=2)

print("=== エージェントの処理過程 ===")
print(full_log)

print("\n=== ✅ 最終回答（第2稿） ===")
print(final_answer)

## **3. 何が起きたのか？（処理過程の分析）**

`=== エージェントの処理過程 ===` のログを読むと、以下の流れがはっきりと確認できます。

1. **Writer (Round 1)**: クエリ通りに、そつなくまとまった「空飛ぶクルマ」の解説（初稿）を作成しました。
2. **Editor (Round 1)**: プロンプトの指示に従い、初稿に対して「具体的な活用シーンの追加」と「ワクワクするトーン」を要求しました。
3. **Writer (Round 2)**: Editor の厳しいフィードバックを受け取り、表現を豊かにし、具体的な利用シーンを盛り込んだ「第2稿」を作成しました。
4. **Editor (Round 2)**: 要求が満たされたことを確認し、「誤りなし」として記事を承認し、ループが終了しました。

これが **Iterative Refinement（反復的改善）** と呼ばれる強力な手法です。
1回のプロンプト（Zero-shot）で最初から完璧なトーンと構成を指定するのは難航しがちですが、**「書かせる役」と「文句を言う役」を分ける**ことで、LLMは客観的な視点を獲得し、劇的に出力の質を高めることができます。

---
### **まとめ**
- 現代の優秀なLLMを用いたエージェントループは、「ミスの修正」だけでなく**「表現や論理のブラッシュアップ（壁打ち相手）」**として絶大な効果を発揮します。
- Critic に**「別のペルソナ（読者代表、編集長、セキュリティ専門家など）」**を与えることで、システム全体が多角的な視点を持つようになります。
- Critic に自身の状態（初回か、2回目か）を認識させることで、無限ループを防ぎつつ確実な改善を引き出すことができます。